# Experimental: local Gemma 3 1B (Stage-2 SLM)

Defaults come from `local_llm/config.py` (`MODEL_ID=gemma3:1b`, bartowski QAT Q4_K_M).
Staging runtime bakes the same GGUF via Dockerfile `ensure_model()` and runs with `LOCAL_LLM_FORCE_CPU=1`.

Run cells top-to-bottom so the model stays loaded. Use the **Latency** section before shipping the 1B switch.

| API | Purpose |
|---|---|
| `ensure_model` | download GGUF into `local_llm/models/` |
| `default_model_path` / `MODELS_DIR` / `MODEL_ID` | paths + model id |
| `detect_runtime` | Mac Metal / Linux CUDA-or-CPU knobs |
| `LocalGemma` | low-level engine: `.chat`, `.complete`, `.yes_no_logit_margin` |
| `LocalLLMClient` | preferred import for other modules |

In [ ]:
from pathlib import Path
import sys
import time

# Repo root on sys.path when the notebook kernel cwd is local_llm/ or repo root.
ROOT = Path.cwd().resolve()
if (ROOT / "local_llm").is_dir():
    pass
elif (ROOT.parent / "local_llm").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from local_llm import (
    LocalGemma,
    LocalLLMClient,
    MODEL_ID,
    MODELS_DIR,
    default_model_path,
    ensure_model,
)
from local_llm.config import HF_REPO, MODEL_FILENAME, QUANT
from local_llm.engine import detect_runtime

print("MODEL_ID", MODEL_ID)
print("HF_REPO", HF_REPO)
print("QUANT", QUANT)
print("MODEL_FILENAME", MODEL_FILENAME)
print("MODELS_DIR", MODELS_DIR)
print("default_model_path", default_model_path())

## `ensure_model`
Downloads the QAT GGUF (~770 MB for Q4_K_M) into the repo folder if missing. Safe to re-run.

In [ ]:
model_path = ensure_model()
print(model_path)
print(f"{model_path.stat().st_size / (1024 * 1024):.1f} MB")

## `detect_runtime`
Platform tuning used when loading llama.cpp (Metal / CUDA / CPU).
Staging Fargate sets `LOCAL_LLM_FORCE_CPU=1` → backend `cpu`, `n_gpu_layers=0`.

In [ ]:
detect_runtime()

## `LocalGemma` — load once
Reuse this `llm` in the cells below. Time this once; cold load dominates first request after deploy.

In [ ]:
t0 = time.perf_counter()
llm = LocalGemma()  # ensure=True downloads if needed
load_s = time.perf_counter() - t0
print(f"load_s={load_s:.3f}")
llm.model_id, llm.model_path, llm.runtime

## Latency — Stage-2 path (what staging pays for)

Stage-2 uses `yes_no_logit_margin` (next-token YES−NO logits), not free-text chat.
Measure that first. Optionally force CPU to approximate Fargate.

In [ ]:
import os

# Staging-shaped few-shot YES/NO prompt (short on purpose).
STAGE2_PROMPT = """Decide if the query matches the recall intent. Answer YES or NO.

Recall: remember facts the user previously asked about.
Examples:
Q: remind me what name was on yesterday's email?
A: YES
Q: write a script to scrape this website
A: NO
Q: when did we choose MongoDB again?
A: YES
Q: tell me a joke about a programmer
A: NO

Q: what API design choice did we settle on?
A:"""

N_WARMUP = 2
N_ITERS = 20

for _ in range(N_WARMUP):
    llm.yes_no_logit_margin(STAGE2_PROMPT)

times = []
margins = []
for _ in range(N_ITERS):
    t0 = time.perf_counter()
    margin = llm.yes_no_logit_margin(STAGE2_PROMPT)
    times.append(time.perf_counter() - t0)
    margins.append(margin)

times_ms = [t * 1000 for t in times]
print({
    "model_id": llm.model_id,
    "backend": llm.runtime.get("backend"),
    "n_threads": llm.runtime.get("n_threads"),
    "n_gpu_layers": llm.runtime.get("n_gpu_layers"),
    "force_cpu_env": os.environ.get("LOCAL_LLM_FORCE_CPU"),
    "iters": N_ITERS,
    "yes_no_ms_p50": round(sorted(times_ms)[len(times_ms) // 2], 2),
    "yes_no_ms_p95": round(sorted(times_ms)[int(len(times_ms) * 0.95) - 1], 2),
    "yes_no_ms_mean": round(sum(times_ms) / len(times_ms), 2),
    "margin_last": round(margins[-1], 3),
    "margin_mean": round(sum(margins) / len(margins), 3),
})

### Optional: staging-like CPU reload

On Mac this forces CPU (no Metal). Restart the kernel first if you already loaded with Metal,
or close the current engine, set the env, and construct a new `LocalGemma`.

In [ ]:
# Uncomment to benchmark Fargate-shaped CPU path on this host.
# llm.close()
# os.environ["LOCAL_LLM_FORCE_CPU"] = "1"
# os.environ["LOCAL_LLM_N_GPU_LAYERS"] = "0"
# t0 = time.perf_counter()
# llm = LocalGemma()
# print(f"cpu_load_s={time.perf_counter() - t0:.3f}", llm.runtime)
# then re-run the Stage-2 latency cell above.
print("skipped — uncomment to force CPU reload")

### Chat / complete latency (sanity, not Stage-2)

In [ ]:
def bench_chat(n: int = 10, max_tokens: int = 16) -> dict:
    msgs = [{"role": "user", "content": "Reply with exactly: ok"}]
    for _ in range(2):
        llm.chat(msgs, max_tokens=max_tokens, temperature=0.0)
    times_ms = []
    for _ in range(n):
        t0 = time.perf_counter()
        llm.chat(msgs, max_tokens=max_tokens, temperature=0.0)
        times_ms.append((time.perf_counter() - t0) * 1000)
    return {
        "chat_ms_p50": round(sorted(times_ms)[len(times_ms) // 2], 2),
        "chat_ms_p95": round(sorted(times_ms)[int(len(times_ms) * 0.95) - 1], 2),
        "chat_ms_mean": round(sum(times_ms) / len(times_ms), 2),
        "max_tokens": max_tokens,
        "iters": n,
    }

bench_chat()

## `LocalGemma.complete`
Raw text completion (no chat template).

In [ ]:
result = llm.complete(
    "Write one short sentence about staging deploys:\n",
    max_tokens=64,
    temperature=0.7,
)
print(result.text)
print(
    {
        "finish_reason": result.finish_reason,
        "prompt_tokens": result.prompt_tokens,
        "completion_tokens": result.completion_tokens,
        "total_tokens": result.total_tokens,
    }
)

## `LocalGemma.chat`
Chat messages (system / user / assistant). This is the usual call shape for other modules.

In [ ]:
result = llm.chat(
    [
        {"role": "system", "content": "Answer in one short sentence."},
        {"role": "user", "content": "what can you do?"},
    ],
    max_tokens=96,
    temperature=0.7,
)
print(result.text)
print(result.prompt_tokens, result.completion_tokens, result.finish_reason)

## `LocalLLMClient` (preferred for other files)
Thin wrapper over `LocalGemma`. Pass a shared engine so the model is not loaded twice.

In [ ]:
client = LocalLLMClient(in_process=True, engine=llm)
client.model, client.in_process

## Optional cleanup
Free the loaded model when done experimenting.

In [ ]:
llm.close()
print("closed")